# 06 · Capstone: A Full ETL Pipeline

Time to combine everything into one small but **production-shaped** pipeline:

**Extract** raw CSV/JSONL → **Validate** with pydantic → **Transform** with
pandas → **Load** to Parquet + SQLite, with **logging**, **idempotency**, and a
**test**. This is the shape of real data-engineering code.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

## 1. Logging setup

Every run should be observable. We configure a logger once and use it through
each stage.

In [ ]:
import logging, sys
logger = logging.getLogger('capstone')
logger.handlers.clear()
logger.setLevel(logging.INFO)
h = logging.StreamHandler(sys.stdout)
h.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-5s | %(message)s',
                                 datefmt='%H:%M:%S'))
logger.addHandler(h)
logger.info('pipeline configured')

## 2. Extract

Read the raw orders and customers. Extraction just *gets bytes into memory* —
no business logic yet.

In [ ]:
import pandas as pd

def extract():
    orders = pd.read_csv(RAW / 'orders.csv', parse_dates=['order_ts'])
    customers = pd.read_csv(RAW / 'customers.csv')
    logger.info('extracted orders=%d customers=%d', len(orders), len(customers))
    return orders, customers

orders, customers = extract()
orders.head(3)

## 3. Validate

Enforce data quality at the boundary. We validate order rows with pydantic,
separating good records from a **dead-letter** list — bad data never silently
enters the warehouse.

In [ ]:
from pydantic import BaseModel, ValidationError, field_validator

class OrderRow(BaseModel):
    order_id: int
    customer_id: int
    status: str
    amount: float

    @field_validator('amount')
    @classmethod
    def non_negative(cls, v):
        if v < 0:
            raise ValueError('amount < 0')
        return v

def validate(orders_df):
    good, dead = [], []
    for rec in orders_df.to_dict('records'):
        try:
            OrderRow(**{k: rec[k] for k in ('order_id','customer_id','status','amount')})
            good.append(rec)
        except ValidationError as e:
            dead.append((rec.get('order_id'), e.errors()[0]['msg']))
    logger.info('validated good=%d dead_letter=%d', len(good), len(dead))
    return pd.DataFrame(good), dead

valid_orders, dead_letter = validate(orders)
print('dead-letter sample:', dead_letter[:3])

## 4. Transform

Clean keys, join, and build the analytics table: completed revenue by country
and month. This is the business logic — the reason the pipeline exists.

In [ ]:
def transform(orders_df, customers_df):
    cust = customers_df.copy()
    cust['country'] = cust['country'].str.strip().str.upper()   # clean keys
    enriched = orders_df.merge(cust[['customer_id', 'country']],
                               on='customer_id', how='left')
    enriched = enriched[enriched['status'] == 'completed'].copy()
    enriched['month'] = pd.to_datetime(enriched['order_ts']).dt.to_period('M').astype(str)
    result = (enriched.groupby(['country', 'month'])
              .agg(orders=('order_id', 'count'),
                   revenue=('amount', 'sum'))
              .round(2).reset_index())
    logger.info('transformed rows=%d', len(result))
    return result

mart = transform(valid_orders, customers)
mart.sort_values('revenue', ascending=False).head()

## 5. Load (idempotent)

Write the result to **Parquet** (for the lake) and **SQLite** (for querying).
**Idempotency** matters: re-running must not duplicate data. We overwrite the
target table/file, so running the pipeline twice yields the same state — a core
production property.

In [ ]:
from sqlalchemy import create_engine

def load(df):
    wh = DATA / 'warehouse'
    wh.mkdir(parents=True, exist_ok=True)
    df.to_parquet(wh / 'revenue_by_country_month.parquet', index=False)
    engine = create_engine(f'sqlite:///{DATA / "retail.db"}')
    df.to_sql('mart_revenue', engine, if_exists='replace', index=False)  # idempotent
    logger.info('loaded rows=%d to parquet + sqlite', len(df))

load(mart)
load(mart)   # run twice on purpose...

check = pd.read_sql('SELECT COUNT(*) AS n FROM mart_revenue', 
                    create_engine(f'sqlite:///{DATA / "retail.db"}'))
print('rows after running load() twice:', int(check['n'][0]), '(no duplication)')

## 6. Orchestrate

A single entry point runs the stages in order — the function a scheduler
(cron, Airflow) would call. Notice how each stage is small, named, and testable.

In [ ]:
def run_pipeline():
    logger.info('=== pipeline start ===')
    orders_df, customers_df = extract()
    valid, dead = validate(orders_df)
    mart_df = transform(valid, customers_df)
    load(mart_df)
    logger.info('=== pipeline done: %d mart rows, %d rejected ===',
                len(mart_df), len(dead))
    return mart_df

final = run_pipeline()
print('\nTop 5 country-months by revenue:')
print(final.sort_values('revenue', ascending=False).head())

## 7. Test the transform

The most valuable test targets the business logic. We verify the transform only
counts completed orders and rolls up correctly — on a tiny, hand-built input.

In [ ]:
import ipytest, pandas as pd
ipytest.autoconfig()

In [ ]:
%%ipytest

def test_transform_only_completed_and_rolls_up():
    orders_df = pd.DataFrame({
        'order_id': [1, 2, 3],
        'customer_id': [10, 10, 20],
        'status': ['completed', 'returned', 'completed'],
        'order_ts': pd.to_datetime(['2024-01-05', '2024-01-06', '2024-01-20']),
        'amount': [100.0, 999.0, 50.0],
    })
    customers_df = pd.DataFrame({
        'customer_id': [10, 20],
        'country': ['us', 'GB '],
    })
    out = transform(orders_df, customers_df)
    # returned order (999) must be excluded; keys cleaned to US/GB
    assert out['revenue'].sum() == 150.0
    assert set(out['country']) == {'US', 'GB'}
    assert out['orders'].sum() == 2

## 🎓 You made it

You built a validated, logged, idempotent, tested ETL pipeline — and along the
way learned Python from variables to production data engineering.

**Where to go next:** move these stages into a `src/` package with real
`tests/`, run it with `uv run python -m pipeline --date ...` (the Python bootcamp's
logging & CLI notebook), swap SQLite for Postgres via the SQLAlchemy engine (the
Python bootcamp's databases notebook), schedule it, and partition the Parquet
output by month (notebook 05). The patterns are identical
at scale.

Happy shipping! 🚀